# Module 1: mRMR from Scratch

## Learning Objectives

By completing this notebook, you will:
1. Implement both MID and MIQ variants of mRMR from scratch
2. Apply mRMR to a real dataset and compare the selected set against a pure MI ranking
3. Visualise the relevance-redundancy trade-off that drives mRMR's greedy selections
4. Understand when redundancy removal changes the selected feature set in practice

## Prerequisites
- Notebook 01: MI deep dive (MI estimation, sklearn MI functions)
- Guide 03: mRMR and Relief

## Estimated Time: 15 minutes

---

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Setup complete.")

## 2. Load and Prepare Data

We use the breast cancer dataset: 30 features derived from cell nuclei measurements, binary target (malignant vs benign). Several features are highly correlated — a perfect testbed for redundancy removal.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

scaler = StandardScaler()
X_sc = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print(f"Breast cancer dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Class balance: {y.sum()} malignant, {(1-y).sum()} benign")

# Feature groups (by name pattern) — shows the redundancy structure
prefixes = ['mean', 'se', 'worst']
base_feats = ['radius', 'texture', 'perimeter', 'area', 'smoothness',
              'compactness', 'concavity', 'concave points', 'symmetry',
              'fractal dimension']
print(f"\nFeature structure: 3 statistics (mean/se/worst) × 10 measurements")
print(f"Strong correlations expected between mean/se/worst of the same measurement")

## 3. Build the MI Infrastructure

mRMR needs two MI computations:
1. **Relevance:** $I(f_j; Y)$ — MI between each feature and the target
2. **Redundancy:** $I(f_j; f_i)$ — MI between features (full pairwise matrix)

The pairwise feature MI matrix is computed once and reused throughout the greedy selection.

In [ ]:
def compute_relevance(X: pd.DataFrame, y: np.ndarray,
                      n_neighbors: int = 5, random_state: int = 42) -> pd.Series:
    """
    Compute MI(f_j, Y) for all features.
    Returns a Series indexed by feature name, sorted descending.
    """
    scores = mutual_info_classif(X, y, n_neighbors=n_neighbors,
                                  random_state=random_state)
    return pd.Series(scores, index=X.columns).sort_values(ascending=False)


def compute_pairwise_mi(X: pd.DataFrame, n_neighbors: int = 5,
                         random_state: int = 42) -> pd.DataFrame:
    """
    Compute symmetric MI matrix among all features.

    For each feature f_i, treat it as the 'target' and compute
    MI with all other features using mutual_info_classif.
    MI is symmetric: I(X;Y) = I(Y;X).

    Returns
    -------
    pd.DataFrame of shape (n_features, n_features)
    """
    cols = list(X.columns)
    p = len(cols)
    mi_mat = np.zeros((p, p))

    for i, target_col in enumerate(cols):
        # Use mutual_info_classif treating target_col as a discrete-ish target
        # (after StandardScaler it's continuous, but the estimator handles this)
        other_cols = [c for c in cols if c != target_col]
        other_idx  = [cols.index(c) for c in other_cols]

        # Discretise target_col into bins for "classification" MI
        target_vals = X[target_col].values
        target_binned = pd.cut(target_vals, bins=10, labels=False)
        target_binned = np.nan_to_num(target_binned.astype(float)).astype(int)

        scores = mutual_info_classif(
            X[other_cols], target_binned,
            n_neighbors=n_neighbors, random_state=random_state
        )

        for k, j in enumerate(other_idx):
            mi_mat[i, j] = scores[k]
            mi_mat[j, i] = scores[k]  # symmetry

    return pd.DataFrame(mi_mat, index=cols, columns=cols)


print("Computing relevance scores...")
relevance = compute_relevance(X_sc, y)
print("Top 5 features by MI with target:")
print(relevance.head().round(4))

print("\nComputing pairwise MI matrix (this takes ~30 seconds)...")
mi_matrix = compute_pairwise_mi(X_sc)
print(f"Pairwise MI matrix shape: {mi_matrix.shape}")

In [ ]:
# Visualise the pairwise MI matrix as a heatmap
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.eye(mi_matrix.shape[0], dtype=bool)  # hide diagonal for clarity
sns.heatmap(mi_matrix, mask=mask, cmap='YlOrRd', ax=ax,
            xticklabels=True, yticklabels=True, annot=False,
            cbar_kws={'label': 'MI (nats)'})
ax.set_title('Pairwise MI Among Features (Breast Cancer)', fontsize=13, fontweight='bold')
plt.xticks(rotation=90, fontsize=6.5)
plt.yticks(rotation=0, fontsize=6.5)
plt.tight_layout()
plt.savefig('pairwise_mi_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

# Identify the most redundant pairs
mi_upper = mi_matrix.where(np.triu(np.ones(mi_matrix.shape), k=1).astype(bool))
mi_stack = mi_upper.stack().sort_values(ascending=False)
print("\nTop 5 most redundant feature pairs (highest inter-feature MI):")
print(mi_stack.head().round(4))

## 4. mRMR: Full Implementation

With the relevance scores and pairwise MI matrix in hand, the greedy mRMR selection is straightforward. The key loop: at each step, score all remaining features by their relevance minus (or divided by) their average redundancy with the already-selected set.

In [ ]:
def mrmr_select(relevance: pd.Series, mi_matrix: pd.DataFrame,
                n_features: int, variant: str = 'MID') -> tuple:
    """
    Greedy mRMR feature selection.

    Parameters
    ----------
    relevance : Series
        MI(f_j, Y) for each feature.
    mi_matrix : DataFrame
        Pairwise MI among features, shape (p, p).
    n_features : int
        Number of features to select.
    variant : 'MID' or 'MIQ'

    Returns
    -------
    selected : list of str
        Feature names in selection order.
    score_history : list of dict
        Per-step scores for visualisation (relevance, redundancy, mrmr_score).
    """
    all_features = list(relevance.index)
    remaining    = all_features.copy()
    selected     = []
    score_history = []

    # Step 0: select feature with highest relevance (no redundancy yet)
    first_feature = relevance.idxmax()
    selected.append(first_feature)
    remaining.remove(first_feature)
    score_history.append({
        'step': 1,
        'selected': first_feature,
        'relevance': relevance[first_feature],
        'redundancy': 0.0,
        'mrmr_score': relevance[first_feature],
    })

    # Greedy steps 1 to n_features - 1
    for step in range(1, n_features):
        step_scores = {}
        step_details = {}

        for feat in remaining:
            rel = relevance[feat]
            # Average MI with already-selected features
            red = mi_matrix.loc[feat, selected].mean()

            if variant == 'MID':
                score = rel - red
            elif variant == 'MIQ':
                score = rel / (red + 1e-10)
            else:
                raise ValueError(f"Unknown variant '{variant}'. Use 'MID' or 'MIQ'.")

            step_scores[feat]  = score
            step_details[feat] = (rel, red, score)

        best_feat = max(step_scores, key=step_scores.get)
        rel_b, red_b, score_b = step_details[best_feat]

        selected.append(best_feat)
        remaining.remove(best_feat)
        score_history.append({
            'step': step + 1,
            'selected': best_feat,
            'relevance': rel_b,
            'redundancy': red_b,
            'mrmr_score': score_b,
        })

    return selected, score_history


# Run both MID and MIQ variants
n_select = 10
selected_mid, history_mid = mrmr_select(relevance, mi_matrix, n_select, 'MID')
selected_miq, history_miq = mrmr_select(relevance, mi_matrix, n_select, 'MIQ')

# Pure MI ranking (top-k by relevance, no redundancy removal)
top_mi = relevance.head(n_select).index.tolist()

print(f"Top {n_select} features by pure MI:   {top_mi}")
print(f"mRMR-MID selected features: {selected_mid}")
print(f"mRMR-MIQ selected features: {selected_miq}")
print(f"\nMI ∩ MID: {set(top_mi) & set(selected_mid)}")
print(f"Features removed by MID redundancy: {set(top_mi) - set(selected_mid)}")
print(f"Features added by MID (were outside top-10 MI): {set(selected_mid) - set(top_mi)}")

## 5. Visualise the Relevance-Redundancy Trade-Off

At each step, mRMR weighs two competing signals:
- **Relevance** $I(f_j; Y)$ — high is good
- **Redundancy** $\frac{1}{|S|} \sum_{f_i \in S} I(f_j; f_i)$ — low is good

The selection plot shows how redundancy grows as more features are added, and how mRMR's score accounts for this.

In [ ]:
history_df = pd.DataFrame(history_mid)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Relevance and redundancy at each selection step
steps = history_df['step'].values
axes[0].bar(steps, history_df['relevance'], label='Relevance I(f;Y)', color='#2ecc71', alpha=0.8)
axes[0].bar(steps, -history_df['redundancy'], label='−Redundancy (penalty)',
             color='#e74c3c', alpha=0.8)
axes[0].plot(steps, history_df['mrmr_score'], 'k.-', linewidth=2,
              markersize=8, label='mRMR-MID score')
axes[0].axhline(0, color='black', linewidth=0.8, alpha=0.5)
axes[0].set_xticks(steps)
axes[0].set_xticklabels([h['selected'][:12] for h in history_mid],
                          rotation=45, ha='right', fontsize=7)
axes[0].set_ylabel('Value (nats)')
axes[0].set_title('mRMR-MID: Relevance vs Redundancy Trade-Off')
axes[0].legend(fontsize=8)

# Subplot 2: MID vs MIQ selection paths as rank comparison
# Show where the two variants diverge
all_features_list = list(relevance.index)
mi_rank = {f: i+1 for i, f in enumerate(relevance.index)}

for step_i, (feat_mid, feat_miq) in enumerate(
    zip([h['selected'] for h in history_mid],
        [h['selected'] for h in history_miq]), start=1):

    axes[1].scatter(step_i, mi_rank[feat_mid], color='#3498db', s=80, zorder=5)
    axes[1].scatter(step_i, mi_rank[feat_miq], color='#e67e22', s=80, zorder=5, marker='^')

    if feat_mid != feat_miq:
        axes[1].annotate(f'MID: {feat_mid[:10]}\nMIQ: {feat_miq[:10]}',
                          (step_i, max(mi_rank[feat_mid], mi_rank[feat_miq]) + 0.5),
                          fontsize=6, ha='center', color='black')

axes[1].plot(range(1, n_select+1), [h for h in range(1, n_select+1)],
              'k--', alpha=0.3, label='Pure MI (top-k)')
axes[1].scatter([], [], color='#3498db', label='MID selected', s=80)
axes[1].scatter([], [], color='#e67e22', marker='^', label='MIQ selected', s=80)
axes[1].set_xlabel('Selection step')
axes[1].set_ylabel('Original MI rank (1 = highest MI)')
axes[1].set_title('MID vs MIQ: Which MI-rank Feature Gets Selected?')
axes[1].legend(fontsize=8)
axes[1].invert_yaxis()

plt.suptitle('mRMR Greedy Selection: Seeing the Trade-Off',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('mrmr_selection_tradeoff.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Does Redundancy Removal Actually Help?

We evaluate three feature sets by cross-validated logistic regression accuracy:
1. Pure top-$k$ MI features (no redundancy removal)
2. mRMR-MID features
3. mRMR-MIQ features

If the breast cancer features are redundant, removing them should not hurt accuracy while potentially improving generalisation.

In [ ]:
def evaluate_feature_set(X_data: pd.DataFrame, y_data: np.ndarray,
                          feature_names: list, cv: int = 5) -> dict:
    """
    Evaluate a feature set via cross-validated logistic regression.
    Returns mean CV accuracy and std.
    """
    clf = LogisticRegression(max_iter=2000, random_state=42)
    scores = cross_val_score(clf, X_data[feature_names], y_data,
                              cv=cv, scoring='accuracy')
    return {'mean_cv_acc': scores.mean(), 'std_cv_acc': scores.std(),
            'n_features': len(feature_names)}


feature_sets = {
    'All features (30)':     list(X_sc.columns),
    f'Top-{n_select} MI':   top_mi,
    f'mRMR-MID ({n_select})': selected_mid,
    f'mRMR-MIQ ({n_select})': selected_miq,
}

eval_results = {}
for name, feats in feature_sets.items():
    eval_results[name] = evaluate_feature_set(X_sc, y, feats)
    print(f"{name:30s}: "
          f"acc = {eval_results[name]['mean_cv_acc']:.4f} "
          f"± {eval_results[name]['std_cv_acc']:.4f}")

In [ ]:
# Sweep: compare MI vs mRMR-MID at different k values
k_values = list(range(2, min(21, len(X.columns) + 1)))
acc_mi  = []
acc_mid = []

for k in k_values:
    top_k_mi = relevance.head(k).index.tolist()
    sel_mid, _ = mrmr_select(relevance, mi_matrix, k, 'MID')

    acc_mi.append(evaluate_feature_set(X_sc, y, top_k_mi)['mean_cv_acc'])
    acc_mid.append(evaluate_feature_set(X_sc, y, sel_mid)['mean_cv_acc'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_values, acc_mi,  'o-', color='#e74c3c', linewidth=2, label='Top-k MI (no redundancy removal)')
ax.plot(k_values, acc_mid, 's-', color='#2ecc71', linewidth=2, label='mRMR-MID')
ax.set_xlabel('Number of features selected (k)')
ax.set_ylabel('5-fold CV accuracy (logistic regression)')
ax.set_title('mRMR-MID vs Pure MI Ranking: Accuracy vs Feature Count')
ax.legend()
ax.set_ylim(0.90, 0.99)
plt.tight_layout()
plt.savefig('mrmr_vs_mi_accuracy.png', dpi=120, bbox_inches='tight')
plt.show()

# Find k where mRMR beats pure MI most
diff = np.array(acc_mid) - np.array(acc_mi)
best_k = k_values[diff.argmax()]
print(f"mRMR-MID beats pure MI by {diff.max():.4f} at k={best_k} features.")

## 7. Exercise: mRMR with a Different Redundancy Measure

**Task:** Implement a variant of mRMR that uses Pearson correlation instead of MI for the redundancy term:

$$\phi_\text{hybrid}(f_j) = I(f_j; Y) - \frac{1}{|S_m|} \sum_{f_i \in S_m} |r(f_j, f_i)|$$

This hybrid uses MI for relevance (captures nonlinear dependencies) but Pearson for redundancy (much cheaper to compute).

Compare the hybrid's selected set with full mRMR-MID on the breast cancer dataset.

**Hints:**
- Compute the full Pearson correlation matrix using `X_sc.corr()`
- Modify `mrmr_select` to accept an optional `redundancy_matrix` parameter
- Use `abs()` on Pearson r (both positive and negative correlations indicate redundancy)

In [ ]:
# YOUR CODE HERE
# ---------------

def mrmr_hybrid(relevance: pd.Series, X: pd.DataFrame, y: np.ndarray,
                n_features: int) -> list:
    """
    mRMR using MI for relevance and |Pearson r| for redundancy.

    Parameters
    ----------
    relevance : Series — MI(f_j, Y)
    X : DataFrame — feature matrix (for correlation computation)
    y : array — target
    n_features : int

    Returns
    -------
    list of str — selected feature names
    """
    # TODO: compute Pearson correlation matrix
    # TODO: implement greedy loop using |r| as redundancy
    pass


# selected_hybrid = mrmr_hybrid(relevance, X_sc, y, n_select)
# print("Hybrid selected:", selected_hybrid)
# print("Full MID selected:", selected_mid)
# print("Agreement:", set(selected_hybrid) & set(selected_mid))

In [ ]:
# Reference solution
def mrmr_hybrid_solution(relevance: pd.Series, X: pd.DataFrame, y: np.ndarray,
                          n_features: int) -> list:
    """
    mRMR with MI relevance and |Pearson r| redundancy.
    Faster than full mRMR because correlation is O(n) per pair vs O(n log n) for KSG.
    """
    # Full pairwise absolute Pearson correlation (cheap)
    corr_mat = X.corr().abs()

    remaining = list(relevance.index)
    selected  = [relevance.idxmax()]
    remaining.remove(selected[0])

    for _ in range(n_features - 1):
        scores = {}
        for feat in remaining:
            rel = relevance[feat]
            red = corr_mat.loc[feat, selected].mean()  # avg |r| with selected
            scores[feat] = rel - red
        best = max(scores, key=scores.get)
        selected.append(best)
        remaining.remove(best)

    return selected


selected_hybrid = mrmr_hybrid_solution(relevance, X_sc, y, n_select)
print(f"Hybrid (MI relevance + |r| redundancy): {selected_hybrid}")
print(f"Full mRMR-MID:                          {selected_mid}")
print(f"Agreement: {len(set(selected_hybrid) & set(selected_mid))}/{n_select} features")

acc_hybrid = evaluate_feature_set(X_sc, y, selected_hybrid)['mean_cv_acc']
acc_mid_10 = evaluate_feature_set(X_sc, y, selected_mid)['mean_cv_acc']
print(f"\nCV accuracy — Hybrid: {acc_hybrid:.4f} | Full mRMR-MID: {acc_mid_10:.4f}")

## 8. Summary

### Key Takeaways

1. **mRMR adds redundancy awareness to MI filtering.** The greedy algorithm penalises features for sharing information with already-selected features.

2. **MID vs MIQ:** MID (subtract redundancy) is more stable when features have similar relevance. MIQ (divide by redundancy) is more aggressive and can diverge when redundancy is near zero.

3. **The pairwise MI matrix is the bottleneck.** It costs $O(p^2)$ MI computations. For $p > 1000$, consider replacing with absolute Pearson correlation (the hybrid approach) — faster and often achieves similar results.

4. **Redundancy removal helps most at small $k$.** When selecting very few features, removing redundant ones matters most. At large $k$, most useful features are eventually included anyway.

### What's Next

Notebook 03 — Dependence measures comparison: how Pearson, Spearman, distance correlation, and HSIC perform on the same dataset.

### Additional Resources
- Peng, H. et al. (2005). *Feature selection based on mutual information.* IEEE TPAMI.
- mRMR Python library: https://github.com/smazzanti/mrmr (faster production implementation)